# Event Labeling

## Process the Data

- **Purpose:** Create triple-barrier direction labels from the integrated point-in-time feature schema.
- **Settings:** `return_horizon=1000` bars; `ewma_span=100`; `vertical_barrier=1000` bars; `pt_sl=(1.0, 1.0)`; `min_target=development 25th percentile`; `min_class_frequency=0.10`.
- **Data:** The integrated development and holdout feature rows produce the persisted labeled-event artifact.
- **Decision:** Fit thresholds on development only, remove rare labels, and purge development events that cross the fixed holdout boundary.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]

from src.data_preprocessing.event_labeling import build_labeled_event_data

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
candidate_path = event_dir / f"aapl_news_candidate_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"

candidate_split = pd.read_parquet(candidate_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")


In [2]:
model_data, partition_manifest = build_labeled_event_data(candidate_split, dollar_bars)

event_dir.mkdir(parents=True, exist_ok=True)
model_data.to_parquet(event_path, index=False)
partition_manifest.to_parquet(partition_path, index=False)
print(event_path)
print(partition_path)


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_primary_model_2025-01-01_2025-12-31.parquet
/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_labeled_split_2025-01-01_2025-12-31.parquet


## Take a Quick Look at the Data Structure

- **Purpose:** Inspect development class balance, numeric ranges, and the persisted event schema.
- **Settings:** No analytical parameters; read-only inspection.
- **Data:** Inspect only the development rows of the persisted 60-column event artifact.
- **Decision:** Leave labels, holdout outcomes, and the artifact unchanged.

In [3]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = model_data[model_data["event_start"].isin(development_starts)]
development_data.head()


,event_start,symbol,event_end,vertical_barrier,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
0,2025-01-13 14:30:01.329809+00:00,AAPL,2025-01-13 14:34:17.401727+00:00,2025-01-16 18:19:25.547320+00:00,0.006302,-0.006778,-1,0.796031,2.001021,-0.0918,...,241.9314,236.8492,9.60,0.8168,242.9775,241.3440,239.7104,242.800,237.9350,233.070
1,2025-01-16 14:30:01.488226+00:00,AAPL,2025-01-16 14:34:43.996829+00:00,2025-01-21 17:48:33.046371+00:00,0.004690,-0.004820,-1,0.168639,2.032826,-7.1295,...,237.7111,237.5796,0.42,0.1261,237.9677,237.7155,237.4634,237.840,237.6300,237.420
2,2025-01-17 14:30:01.792073+00:00,AAPL,2025-01-17 14:39:01.069235+00:00,2025-01-22 20:42:17.531895+00:00,0.006821,-0.007032,-1,-0.898147,2.033963,-7.4341,...,228.4782,226.5652,3.92,0.3121,229.3303,228.7060,228.0818,232.115,230.1125,228.110
3,2025-01-17 17:20:53.298606+00:00,AAPL,2025-01-21 14:30:00.854179+00:00,2025-01-23 16:19:31.331783+00:00,0.005790,-0.025297,-1,0.013412,2.020683,0.2443,...,229.7761,229.4412,0.17,0.2114,230.2595,229.8367,229.4138,230.110,229.8025,229.495
4,2025-01-21 14:30:00.854179+00:00,AAPL,2025-01-21 14:34:53.478136+00:00,2025-01-23 19:56:48.170064+00:00,0.005329,-0.007594,-1,-0.297837,1.994448,1.3457,...,229.3857,226.1968,6.15,0.5650,230.1386,229.0086,227.8786,230.010,226.9350,223.860


In [4]:
development_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 60 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   event_start                               157 non-null    datetime64[us, UTC]
 1   symbol                                    157 non-null    str                
 2   event_end                                 157 non-null    datetime64[us, UTC]
 3   vertical_barrier                          157 non-null    datetime64[us, UTC]
 4   target_return                             157 non-null    float64            
 5   raw_return                                157 non-null    float64            
 6   direction_label                           157 non-null    int8               
 7   mean_sentiment_score                      157 non-null    float64            
 8   fractionally_differenced_log_close        157 non-null    float64      

In [5]:
development_data["direction_label"].value_counts()


direction_label
 1    86
-1    71
Name: count, dtype: int64

In [6]:
development_data.select_dtypes(include="number").describe()


,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,Advancers - Decliners,On-Balance Volume,Accumulation/Distribution Line,Chaikin Oscillator,...,Bollinger Band Middle,Bollinger Band Lower,True Range,Average True Range,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower
count,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,1.570000e+02,1.570000e+02,157.000000,...,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000,157.000000
mean,0.007388,0.001543,0.095541,0.075131,1.996528,0.273489,216.404395,1.308754e+06,2.247826e+06,-490.556334,...,216.724283,215.778974,1.389299,0.319934,217.344983,216.705121,216.065261,217.670000,216.605685,215.541369
std,0.003573,0.014706,0.998611,0.499816,0.029666,2.943072,16.441952,4.285442e+05,9.536545e+05,3753.890812,...,16.458890,16.471781,2.633336,0.206330,16.466644,16.445894,16.435486,16.575875,16.449440,16.432053
min,0.003763,-0.074197,-1.000000,-0.962507,1.909065,-9.475200,171.930000,1.165390e+05,1.994439e+05,-10572.948200,...,172.535400,171.810900,0.080000,0.077100,173.579300,172.547900,171.516400,173.460000,171.040000,168.620000
25%,0.004621,-0.006395,-1.000000,-0.233396,1.974744,-0.198800,203.335000,1.049905e+06,1.890132e+06,-2819.630700,...,203.143900,202.866800,0.220000,0.211400,203.688600,203.049100,202.794100,204.540000,203.745000,202.960000
50%,0.006213,0.004471,1.000000,0.053386,1.989970,0.481800,211.405000,1.275080e+06,2.104888e+06,-430.804500,...,212.425400,210.795600,0.365000,0.261400,212.760600,212.025300,211.081000,212.865000,211.122500,210.650000
75%,0.009455,0.007595,1.000000,0.414660,2.021487,1.737900,230.650000,1.509145e+06,2.829944e+06,2612.041000,...,230.976800,230.220800,1.220000,0.353200,231.545000,231.055000,230.565000,232.115000,231.112500,229.980000
max,0.023920,0.078197,1.000000,0.941092,2.061100,14.341000,254.570000,2.336261e+06,4.134200e+06,9462.076500,...,257.003200,255.587500,17.560000,1.415700,257.632900,256.902200,256.171500,257.430000,255.610000,253.790000


In [7]:
development_data.select_dtypes(include="number").replace([np.inf, -np.inf], np.nan).hist(figsize=(20, 24), bins=30)


array([[<Axes: title={'center': 'target_return'}>,
        <Axes: title={'center': 'raw_return'}>,
        <Axes: title={'center': 'direction_label'}>,
        <Axes: title={'center': 'mean_sentiment_score'}>,
        <Axes: title={'center': 'fractionally_differenced_log_close'}>,
        <Axes: title={'center': 'McClellan Oscillator'}>,
        <Axes: title={'center': 'Advancers - Decliners'}>],
       [<Axes: title={'center': 'On-Balance Volume'}>,
        <Axes: title={'center': 'Accumulation/Distribution Line'}>,
        <Axes: title={'center': 'Chaikin Oscillator'}>,
        <Axes: title={'center': 'New Highs - New Lows'}>,
        <Axes: title={'center': 'Money Flow Index'}>,
        <Axes: title={'center': 'Williams %R'}>,
        <Axes: title={'center': 'Aroon Indicator Up'}>],
       [<Axes: title={'center': 'Aroon Indicator Down'}>,
        <Axes: title={'center': 'Commodity Channel Index'}>,
        <Axes: title={'center': 'Relative Vigor Index'}>,
        <Axes: title={'cen